# Gold Layer: Star Schema

This notebook is the **gold** stage of the NEM medallion pipeline. It reads the three silver tables and reshapes them into a star schema: three dimension tables plus two fact tables, ready for BI and reporting.

Per the project's Python-for-ingestion / SQL-for-transformation split, all transformation logic here is plain SQL. Each section below builds one gold table with `CREATE OR REPLACE TABLE`, followed by a row count and a null check sanity query. Dimensions are built first so that the fact tables can join against their surrogate keys.

Outputs:
- `nem_project.`3_gold`.dim_date`
- `nem_project.`3_gold`.dim_region`
- `nem_project.`3_gold`.dim_facility`
- `nem_project.`3_gold`.fact_facility_power_emissions`
- `nem_project.`3_gold`.fact_region_price_demand`

## 0. Setup

In [0]:
%sql
USE CATALOG nem_project;

## 1. Date Dimension → `nem_project.3_gold.dim_date`

A generated date spine, one row per calendar day from `2025-09-01` to `2026-08-31` (the span covered by the source data). Not derived from any silver table.

- `date_key`: surrogate primary key, `YYYYMMDD` as an `INT`, used by both fact tables.
- Calendar attributes (`year`, `quarter`, `month`, `month_name`, `day_of_month`, `day_of_week`, `day_name`, `week_of_year`) derived directly from `date`.
- `season`: Australian meteorological season (`Summer` = Dec–Feb, `Autumn` = Mar–May, `Winter` = Jun–Aug, `Spring` = Sep–Nov), since this is the NEM.
- `is_weekend`: `TRUE` for Saturday/Sunday.

In [ ]:
%sql
-- ===========================================================================
-- Build gold.dim_date as a generated date spine
-- -------------------------------------------------------------------------
-- Purpose: Provide one row per calendar day covering the source data span,
--          so both fact tables can join to a consistent date dimension.
--
-- Steps:
--   1. Generate the spine with sequence() over the date range, one day at a
--      time, exploded into individual rows.
--   2. Derive the YYYYMMDD surrogate key and every calendar attribute from
--      the spine date.
--   3. Derive season from month using the Australian meteorological
--      convention (Summer = Dec-Feb, Autumn = Mar-May, Winter = Jun-Aug,
--      Spring = Sep-Nov), since this is the NEM.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`3_gold`.dim_date AS
WITH date_spine AS (
  SELECT explode(sequence(DATE'2025-09-01', DATE'2026-08-31', INTERVAL 1 DAY)) AS date
)
SELECT
  -- Surrogate key: YYYYMMDD as an INT
  CAST(date_format(date, 'yyyyMMdd') AS INT) AS date_key,
  date,
  CAST(year(date) AS INT) AS year,
  CAST(quarter(date) AS INT) AS quarter,
  CAST(month(date) AS INT) AS month,
  CAST(date_format(date, 'MMMM') AS STRING) AS month_name,
  CAST(day(date) AS INT) AS day_of_month,
  -- dayofweek(): 1 = Sunday ... 7 = Saturday
  CAST(dayofweek(date) AS INT) AS day_of_week,
  CAST(date_format(date, 'EEEE') AS STRING) AS day_name,
  CAST(weekofyear(date) AS INT) AS week_of_year,
  -- season: Australian meteorological seasons, based on month
  CAST(
    CASE month(date)
      WHEN 12 THEN 'Summer' WHEN 1 THEN 'Summer' WHEN 2 THEN 'Summer'
      WHEN 3 THEN 'Autumn' WHEN 4 THEN 'Autumn' WHEN 5 THEN 'Autumn'
      WHEN 6 THEN 'Winter' WHEN 7 THEN 'Winter' WHEN 8 THEN 'Winter'
      WHEN 9 THEN 'Spring' WHEN 10 THEN 'Spring' WHEN 11 THEN 'Spring'
    END AS STRING
  ) AS season,
  (dayofweek(date) IN (1, 7)) AS is_weekend
FROM date_spine;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`3_gold`.dim_date;

In [ ]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS null_date_key,
  SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
  SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END) AS null_year,
  SUM(CASE WHEN month_name IS NULL THEN 1 ELSE 0 END) AS null_month_name,
  SUM(CASE WHEN day_name IS NULL THEN 1 ELSE 0 END) AS null_day_name,
  SUM(CASE WHEN season IS NULL THEN 1 ELSE 0 END) AS null_season,
  COUNT(DISTINCT date_key) AS distinct_date_keys,
  COUNT(*) AS total_rows
FROM nem_project.`3_gold`.dim_date;

## 2. Region Dimension → `nem_project.3_gold.dim_region`

Built from the distinct `network_region` values found across `2_silver.facilities` and `2_silver.region_price_demand` (a `UNION` of both, since either source could in principle carry a region the other doesn't).

- `region_key`: surrogate primary key via `ROW_NUMBER()`.
- `network_region`: the natural key (e.g. `NSW`).
- `region_name`: the full state name, mapped from `network_region` via a fixed lookup.

In [0]:
%sql
-- ===========================================================================
-- Build gold.dim_region from the distinct regions in silver
-- -------------------------------------------------------------------------
-- Purpose: Provide one row per NEM region, with a surrogate key for the
--          fact tables and a human-readable full name.
--
-- Steps:
--   1. Union the distinct network_region values from facilities and
--      region_price_demand, so no region is missed regardless of source.
--   2. Assign a surrogate region_key via ROW_NUMBER().
--   3. Map network_region to its full state name via a fixed lookup.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`3_gold`.dim_region AS
WITH distinct_regions AS (
  SELECT DISTINCT network_region
  FROM nem_project.`2_silver`.facilities
  WHERE network_region IS NOT NULL

  UNION

  SELECT DISTINCT network_region
  FROM nem_project.`2_silver`.region_price_demand
  WHERE network_region IS NOT NULL
)
SELECT
  -- Surrogate key
  CAST(ROW_NUMBER() OVER (ORDER BY network_region) AS INT) AS region_key,
  CAST(network_region AS STRING) AS network_region,
  -- region_name: full state name, falls back to the raw code if no mapping matches
  CAST(
    CASE network_region
      WHEN 'NSW' THEN 'New South Wales'
      WHEN 'QLD' THEN 'Queensland'
      WHEN 'SA' THEN 'South Australia'
      WHEN 'TAS' THEN 'Tasmania'
      WHEN 'VIC' THEN 'Victoria'
      ELSE network_region
    END AS STRING
  ) AS region_name
FROM distinct_regions;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`3_gold`.dim_region;

In [0]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN region_key IS NULL THEN 1 ELSE 0 END) AS null_region_key,
  SUM(CASE WHEN network_region IS NULL THEN 1 ELSE 0 END) AS null_network_region,
  SUM(CASE WHEN region_name IS NULL THEN 1 ELSE 0 END) AS null_region_name,
  COUNT(DISTINCT network_region) AS distinct_regions
FROM nem_project.`3_gold`.dim_region;

## 3. Facility Dimension → `nem_project.3_gold.dim_facility`

Built from `2_silver.facilities`, joined to `dim_region` on `network_region` to attach the `region_key` foreign key. All other columns are carried through unchanged from silver.

- `facility_key`: surrogate primary key via `ROW_NUMBER()`.
- `facility_code` / `unit_code`: the natural key, carried through from silver.
- `region_key`: foreign key to `dim_region`.

In [0]:
%sql
-- ===========================================================================
-- Build gold.dim_facility from silver.facilities
-- -------------------------------------------------------------------------
-- Purpose: Provide one row per (facility_code, unit_code), with a surrogate
--          key for the fact tables and the region_key foreign key attached.
--
-- Steps:
--   1. Join silver.facilities to dim_region on network_region to resolve
--      the region_key foreign key.
--   2. Assign a surrogate facility_key via ROW_NUMBER().
--   3. Carry through every other facility attribute unchanged from silver.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`3_gold`.dim_facility AS
SELECT
  -- Surrogate key
  CAST(ROW_NUMBER() OVER (ORDER BY f.facility_code, f.unit_code) AS INT) AS facility_key,
  f.facility_code,
  f.unit_code,
  f.facility_name,
  f.fueltech_id,
  f.fueltech_label,
  f.fueltech_category,
  f.status_id,
  f.dispatch_type,
  f.lat,
  f.lng,
  f.capacity_registered,
  f.capacity_maximum,
  f.capacity_storage,
  f.data_first_seen,
  f.data_last_seen,
  f.is_location_missing,
  f.data_is_missing,
  -- Foreign key to dim_region
  r.region_key
FROM nem_project.`2_silver`.facilities AS f
LEFT JOIN nem_project.`3_gold`.dim_region AS r
  ON f.network_region = r.network_region;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`3_gold`.dim_facility;

In [0]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN facility_key IS NULL THEN 1 ELSE 0 END) AS null_facility_key,
  SUM(CASE WHEN facility_code IS NULL THEN 1 ELSE 0 END) AS null_facility_code,
  SUM(CASE WHEN unit_code IS NULL THEN 1 ELSE 0 END) AS null_unit_code,
  SUM(CASE WHEN region_key IS NULL THEN 1 ELSE 0 END) AS null_region_key,
  SUM(CASE WHEN is_location_missing THEN 1 ELSE 0 END) AS flagged_missing_location,
  SUM(CASE WHEN data_is_missing THEN 1 ELSE 0 END) AS flagged_missing_data
FROM nem_project.`3_gold`.dim_facility;

## 4. Facility Power & Emissions Fact → `nem_project.3_gold.fact_facility_power_emissions`

Built from `2_silver.facility_power_emissions`, joined to `dim_facility` on (`facility_code`, `unit_code`) and to `dim_date` on `CAST(time AS DATE) = dim_date.date`. The natural keys are dropped in favour of the resolved surrogate keys; the measures and their missing-value flags carry straight through from silver.

In [0]:
%sql
-- ===========================================================================
-- Build gold.fact_facility_power_emissions from silver.facility_power_emissions
-- -------------------------------------------------------------------------
-- Purpose: One row per facility/unit reading, with facility_key and
--          date_key foreign keys resolved in place of the natural keys.
--
-- Steps:
--   1. Join to dim_facility on (facility_code, unit_code) to resolve
--      facility_key.
--   2. Join to dim_date on CAST(time AS DATE) = dim_date.date to resolve
--      date_key.
--   3. Carry through time, power, emissions, and their missing-value flags
--      unchanged from silver.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`3_gold`.fact_facility_power_emissions AS
SELECT
  -- Foreign keys resolved from the silver natural keys
  df.facility_key,
  dd.date_key,
  pe.time,
  pe.power,
  pe.emissions,
  pe.is_power_missing,
  pe.is_emissions_missing
FROM nem_project.`2_silver`.facility_power_emissions AS pe
LEFT JOIN nem_project.`3_gold`.dim_facility AS df
  ON pe.facility_code = df.facility_code
 AND pe.unit_code = df.unit_code
LEFT JOIN nem_project.`3_gold`.dim_date AS dd
  ON CAST(pe.time AS DATE) = dd.date;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`3_gold`.fact_facility_power_emissions;

In [0]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN facility_key IS NULL THEN 1 ELSE 0 END) AS unmatched_facility_key,
  SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS unmatched_date_key,
  SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS null_time,
  SUM(CASE WHEN is_power_missing THEN 1 ELSE 0 END) AS flagged_missing_power,
  SUM(CASE WHEN is_emissions_missing THEN 1 ELSE 0 END) AS flagged_missing_emissions
FROM nem_project.`3_gold`.fact_facility_power_emissions;

## 5. Region Price & Demand Fact → `nem_project.3_gold.fact_region_price_demand`

Built from `2_silver.region_price_demand`, joined to `dim_region` on `network_region` and to `dim_date` on `CAST(time AS DATE) = dim_date.date`. As with the facility fact, the natural key is replaced by the resolved surrogate keys and the measures carry straight through from silver.

In [0]:
%sql
-- ===========================================================================
-- Build gold.fact_region_price_demand from silver.region_price_demand
-- -------------------------------------------------------------------------
-- Purpose: One row per region/time reading, with region_key and date_key
--          foreign keys resolved in place of the natural key.
--
-- Steps:
--   1. Join to dim_region on network_region to resolve region_key.
--   2. Join to dim_date on CAST(time AS DATE) = dim_date.date to resolve
--      date_key.
--   3. Carry through time, price, and demand unchanged from silver.
-- ===========================================================================

CREATE OR REPLACE TABLE nem_project.`3_gold`.fact_region_price_demand AS
SELECT
  -- Foreign keys resolved from the silver natural key
  dr.region_key,
  dd.date_key,
  rpd.time,
  rpd.price,
  rpd.demand
FROM nem_project.`2_silver`.region_price_demand AS rpd
LEFT JOIN nem_project.`3_gold`.dim_region AS dr
  ON rpd.network_region = dr.network_region
LEFT JOIN nem_project.`3_gold`.dim_date AS dd
  ON CAST(rpd.time AS DATE) = dd.date;

In [0]:
%sql
-- Row count
SELECT COUNT(*) AS row_count
FROM nem_project.`3_gold`.fact_region_price_demand;

In [0]:
%sql
-- Null check sanity query
SELECT
  SUM(CASE WHEN region_key IS NULL THEN 1 ELSE 0 END) AS unmatched_region_key,
  SUM(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS unmatched_date_key,
  SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS null_time,
  SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
  SUM(CASE WHEN demand IS NULL THEN 1 ELSE 0 END) AS null_demand
FROM nem_project.`3_gold`.fact_region_price_demand;

## 6. Summary

Five gold tables were built from the silver layer, all via `CREATE OR REPLACE TABLE` with SQL as the sole transformation language:

#### `dim_date`
* Generated date spine, one row per day from `2025-09-01` to `2026-08-31`.
* `date_key` (`YYYYMMDD`) is the surrogate key shared by both fact tables.
* Includes `season` (Australian meteorological convention) alongside the standard calendar attributes.

#### `dim_region`
* Distinct `network_region` values unioned across `facilities` and `region_price_demand`.
* Added `region_name`, the full state name, via a fixed lookup.

#### `dim_facility`
* One row per (`facility_code`, `unit_code`) from `facilities`.
* Attached `region_key` via a join to `dim_region` on `network_region`.

#### `fact_facility_power_emissions`
* One row per facility/unit reading from `facility_power_emissions`.
* Natural keys replaced by `facility_key` and `date_key` foreign keys.

#### `fact_region_price_demand`
* One row per region/time reading from `region_price_demand`.
* Natural key replaced by `region_key` and `date_key` foreign keys.

Together these five tables form a conventional star schema: two fact tables at the grain of the original silver readings, surrounded by conformed dimensions that both facts share (`dim_date`, and `dim_region` indirectly via `dim_facility`). This is the layer downstream BI tools and reporting queries are going to read from.